# Experiment 1: Quick vs Full Run Comparison

This notebook compares results from:
- **Quick Run** (exp1_quick): Partial completion before cancellation
- **Full Run** (exp1_full): Ongoing full experiment

Both runs test the same methods:
- Med3-TabPFN
- Med3-LoCalPFN
- DenseNet121-3D
- ViT-3D

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Setup plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Paths
repo_root = Path.cwd().parent.parent
quick_csv = repo_root / 'results_from_cluster' / 'combined_benchmarks_summary.csv'
full_csv = repo_root / 'results' / 'experiment1' / 'combined_benchmarks_summary.csv'

print(f"Quick run CSV: {quick_csv}")
print(f"Exists: {quick_csv.exists()}")
print(f"\nFull run CSV: {full_csv}")
print(f"Exists: {full_csv.exists()}")

## Load Results

In [ ]:
# Load CSVs
df_quick = pd.read_csv(quick_csv)
df_full = pd.read_csv(full_csv)

# Add run identifier
df_quick['run'] = 'Quick'
df_full['run'] = 'Full'

print("Quick run shape:", df_quick.shape)
print("Full run shape:", df_full.shape)
print("\nDatasets in Quick run:")
print(df_quick['dataset'].value_counts())
print("\nDatasets in Full run:")
print(df_full['dataset'].value_counts())

## Filter Valid Results (No Errors)

In [ ]:
# Keep only successful runs (no errors, has accuracy)
df_quick_valid = df_quick[df_quick['accuracy'].notna()].copy()
df_full_valid = df_full[df_full['accuracy'].notna()].copy()

print("Valid results (Quick):", len(df_quick_valid))
print("Valid results (Full):", len(df_full_valid))

print("\n=== QUICK RUN: Successful Datasets ===")
print(df_quick_valid[['dataset', 'method', 'accuracy', 'macro_f1', 'roc_auc']].to_string(index=False))

print("\n=== FULL RUN: Successful Datasets ===")
print(df_full_valid[['dataset', 'method', 'accuracy', 'macro_f1', 'roc_auc']].to_string(index=False))

## Compare Common Datasets

In [ ]:
# Find datasets that completed in both runs
quick_datasets = set(df_quick_valid['dataset'].unique())
full_datasets = set(df_full_valid['dataset'].unique())

common_datasets = quick_datasets & full_datasets
only_quick = quick_datasets - full_datasets
only_full = full_datasets - quick_datasets

print(f"Common datasets (in both): {sorted(common_datasets)}")
print(f"Only in Quick: {sorted(only_quick) if only_quick else 'None'}")
print(f"Only in Full: {sorted(only_full) if only_full else 'None'}")

In [ ]:
# Merge on common datasets for comparison
df_comparison = pd.merge(
    df_quick_valid[['dataset', 'method', 'accuracy', 'macro_f1', 'roc_auc']],
    df_full_valid[['dataset', 'method', 'accuracy', 'macro_f1', 'roc_auc']],
    on=['dataset', 'method'],
    how='inner',
    suffixes=('_quick', '_full')
)

# Calculate differences
df_comparison['acc_diff'] = df_comparison['accuracy_full'] - df_comparison['accuracy_quick']
df_comparison['f1_diff'] = df_comparison['macro_f1_full'] - df_comparison['macro_f1_quick']
df_comparison['auc_diff'] = df_comparison['roc_auc_full'] - df_comparison['roc_auc_quick']

print("\n=== COMPARISON: Quick vs Full (Common Datasets) ===")
print(df_comparison.to_string(index=False))

## Visualization: ROC AUC Comparison

In [ ]:
# Create comparison plot for common datasets
if len(df_comparison) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    metrics = ['accuracy', 'macro_f1', 'roc_auc']
    metric_names = ['Accuracy', 'Macro F1', 'ROC AUC']
    
    for ax, metric, name in zip(axes, metrics, metric_names):
        # Create comparison bars
        quick_col = f'{metric}_quick'
        full_col = f'{metric}_full'
        
        x = np.arange(len(df_comparison))
        width = 0.35
        
        ax.bar(x - width/2, df_comparison[quick_col], width, label='Quick', alpha=0.8)
        ax.bar(x + width/2, df_comparison[full_col], width, label='Full', alpha=0.8)
        
        # Labels
        ax.set_xlabel('Dataset - Method')
        ax.set_ylabel(name)
        ax.set_title(f'{name} Comparison: Quick vs Full')
        ax.set_xticks(x)
        labels = [f"{row['dataset']}\n{row['method']}" for _, row in df_comparison.iterrows()]
        ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
        ax.legend()
        ax.grid(axis='y', alpha=0.3)
        
    plt.tight_layout()
    plt.show()
else:
    print("No common datasets to compare.")

## Statistical Summary

In [ ]:
if len(df_comparison) > 0:
    print("=" * 70)
    print("STATISTICAL SUMMARY: Differences (Full - Quick)")
    print("=" * 70)
    
    for metric, name in [('acc', 'Accuracy'), ('f1', 'Macro F1'), ('auc', 'ROC AUC')]:
        diff_col = f'{metric}_diff'
        print(f"\n{name}:")
        print(f"  Mean difference: {df_comparison[diff_col].mean():+.4f}")
        print(f"  Std deviation:   {df_comparison[diff_col].std():.4f}")
        print(f"  Min difference:  {df_comparison[diff_col].min():+.4f}")
        print(f"  Max difference:  {df_comparison[diff_col].max():+.4f}")
        
        # Count improvements/degradations
        improved = (df_comparison[diff_col] > 0).sum()
        degraded = (df_comparison[diff_col] < 0).sum()
        same = (df_comparison[diff_col] == 0).sum()
        print(f"  Improved: {improved} | Degraded: {degraded} | Same: {same}")
else:
    print("No common datasets for statistical comparison.")

## Method-wise Average Performance

In [ ]:
# Average performance per method for each run
print("\n=== QUICK RUN: Average Performance per Method ===")
quick_avg = df_quick_valid.groupby('method')[['accuracy', 'macro_f1', 'roc_auc']].mean()
print(quick_avg.round(4))

print("\n=== FULL RUN: Average Performance per Method ===")
full_avg = df_full_valid.groupby('method')[['accuracy', 'macro_f1', 'roc_auc']].mean()
print(full_avg.round(4))

In [ ]:
# Visualize average performance
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# Combine averages
quick_avg_plot = quick_avg.reset_index()
quick_avg_plot['run'] = 'Quick'
full_avg_plot = full_avg.reset_index()
full_avg_plot['run'] = 'Full'

combined_avg = pd.concat([quick_avg_plot, full_avg_plot])

# Plot ROC AUC (primary metric)
sns.barplot(data=combined_avg, x='method', y='roc_auc', hue='run', ax=ax)
ax.set_xlabel('Method')
ax.set_ylabel('Average ROC AUC')
ax.set_title('Average ROC AUC by Method: Quick vs Full Runs')
ax.set_ylim(0, 1)
ax.legend(title='Run')
ax.grid(axis='y', alpha=0.3)

plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

## Additional Datasets in Full Run

In [ ]:
if only_full:
    print("\n=== ADDITIONAL DATASETS IN FULL RUN ===")
    df_full_extra = df_full_valid[df_full_valid['dataset'].isin(only_full)]
    print(df_full_extra[['dataset', 'method', 'accuracy', 'macro_f1', 'roc_auc']].to_string(index=False))
    
    # Visualize
    if len(df_full_extra) > 0:
        fig, ax = plt.subplots(1, 1, figsize=(10, 6))
        pivot = df_full_extra.pivot(index='dataset', columns='method', values='roc_auc')
        pivot.plot(kind='bar', ax=ax, width=0.8)
        ax.set_xlabel('Dataset')
        ax.set_ylabel('ROC AUC')
        ax.set_title('Performance on Additional Datasets (Full Run Only)')
        ax.set_ylim(0, 1)
        ax.legend(title='Method', bbox_to_anchor=(1.05, 1), loc='upper left')
        ax.grid(axis='y', alpha=0.3)
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.show()
else:
    print("\nNo additional datasets completed in Full run.")

## Error Analysis

In [ ]:
# Check what failed in each run
print("=" * 70)
print("ERROR ANALYSIS")
print("=" * 70)

df_quick_errors = df_quick[df_quick['error'].notna()]
df_full_errors = df_full[df_full['error'].notna()]

print(f"\nQuick run errors: {len(df_quick_errors)} out of {len(df_quick)}")
if len(df_quick_errors) > 0:
    print("\nFailed dataset-method combinations (Quick):")
    for _, row in df_quick_errors.iterrows():
        error_msg = str(row['error']).split('\n')[0]  # First line of error
        print(f"  - {row['dataset']} + {row['method']}: {error_msg}")

print(f"\nFull run errors: {len(df_full_errors)} out of {len(df_full)}")
if len(df_full_errors) > 0:
    print("\nFailed dataset-method combinations (Full):")
    for _, row in df_full_errors.iterrows():
        error_msg = str(row['error']).split('\n')[0]  # First line of error
        print(f"  - {row['dataset']} + {row['method']}: {error_msg}")

## Summary & Conclusions

In [ ]:
print("=" * 70)
print("SUMMARY")
print("=" * 70)

print(f"\nQuick Run:")
print(f"  - Total datasets attempted: {df_quick['dataset'].nunique()}")
print(f"  - Successfully completed: {df_quick_valid['dataset'].nunique()}")
print(f"  - Total valid results: {len(df_quick_valid)}")
print(f"  - Datasets: {sorted(df_quick_valid['dataset'].unique())}")

print(f"\nFull Run:")
print(f"  - Total datasets attempted: {df_full['dataset'].nunique()}")
print(f"  - Successfully completed: {df_full_valid['dataset'].nunique()}")
print(f"  - Total valid results: {len(df_full_valid)}")
print(f"  - Datasets: {sorted(df_full_valid['dataset'].unique())}")

if len(df_comparison) > 0:
    print(f"\nPerformance Difference (Full - Quick) on Common Datasets:")
    print(f"  - Accuracy:  {df_comparison['acc_diff'].mean():+.4f} (avg)")
    print(f"  - Macro F1:  {df_comparison['f1_diff'].mean():+.4f} (avg)")
    print(f"  - ROC AUC:   {df_comparison['auc_diff'].mean():+.4f} (avg)")
    
    if abs(df_comparison['auc_diff'].mean()) < 0.01:
        print("\n✅ Results are highly consistent between Quick and Full runs.")
    elif df_comparison['auc_diff'].mean() > 0:
        print("\n📊 Full run shows slight improvement over Quick run.")
    else:
        print("\n📊 Quick run performed slightly better (possible randomness).")

print("\n" + "=" * 70)